# Домашнее задание № 3

## 1. Найти корень числа (ближайшее целое)

Необходимо написать функцию, которая находит целую часть квадратного корня числа (округление вниз).

Что нужно сделать:

Реализуйте решение с использованием бинарного поиска без использования встроенных функций извлечения корня. Объясните, почему алгоритм работает за $O(log_n)$ и как определяется граница поиска.

In [1]:
def calculate_integer_square_root(n: int) -> int:
    if n == 1:
        return 1

    left, right = 1, n // 2

    while left <= right:
        mid = (left + right) // 2
        mid_squared = mid * mid

        if mid_squared == n:
            return mid
        elif mid_squared < n:
            left = mid + 1
        else:
            right = mid - 1

    return right

In [2]:
test = [-125,-124,-123,1,2,4,8,63,64,65,125,128,255,256,257,1023,1024,1025]

for i in test:
    print(f"sqrt({i}): {calculate_integer_square_root(i)}")

print("-------\nПроверка на граничные случаи:")

print("sqrt(0): ", calculate_integer_square_root(0))
print("sqrt(1): ", calculate_integer_square_root(1))

# print(f"sqrt(-100): {calculate_integer_square_root(-100)}")

sqrt(-125): -63
sqrt(-124): -62
sqrt(-123): -62
sqrt(1): 1
sqrt(2): 1
sqrt(4): 2
sqrt(8): 2
sqrt(63): 7
sqrt(64): 8
sqrt(65): 8
sqrt(125): 11
sqrt(128): 11
sqrt(255): 15
sqrt(256): 16
sqrt(257): 16
sqrt(1023): 31
sqrt(1024): 32
sqrt(1025): 32
-------
Проверка на граничные случаи:
sqrt(0):  0
sqrt(1):  1


### Пояснение

Мы ищем такое число $x$, что:
- $x * x <= n$
- $(x + 1) * (x + 1) > n$

Это и есть целая часть квадратного корня.


Каждый шаг бинарного поиска отбрасывает примерно половину вариантов. Поэтому алгоритм работает за $O(log_n)$.

## 2. Очень лёгкая задача (два ксерокса)

Как быстро можно сделать N копий документа, если есть два ксерокса, каждый копирует с разной скоростью — x и y минут за копию?

Что нужно сделать:
Реализуйте алгоритм нахождения минимального времени, за которое будет сделано N копий. Используйте бинарный поиск по времени и поясните, как проверяется условие достаточности времени.

In [3]:
def min_time_for_copies(n: int, x: int, y: int) -> int:
    if n <= 0:
        return 0
    if n == 1:
        return min(x, y)

    first_copy_time = min(x, y)
    n_remaining = n - 1

    left = 0

    right = min(x, y) * n_remaining

    while left <= right:
        mid = (left + right) // 2
        if (mid // x) + (mid // y) >= n_remaining:
            right = mid - 1
        else:
            left = mid + 1
    return first_copy_time + left


In [4]:
print("n - кол-во копий \nx - время на первом ксероксе \ny - время на втором ксероксе\n")
print("n,x,y:     ", "min t")
print("-----------------")
print("0,3,5:     ",min_time_for_copies(0, 3, 5))
print("1,3,5:     ",min_time_for_copies(1, 3, 5))
print("1,5,3:     ",min_time_for_copies(1, 5, 3))
print("2,4,4:     ",min_time_for_copies(2, 4, 4))
print("3,4,4:     ",min_time_for_copies(3, 4, 4))
print("5,2,2:     ",min_time_for_copies(5, 2, 2))
print("4,1,100:   ",min_time_for_copies(4, 1, 100))
print("3,2,3:     ",min_time_for_copies(3, 2, 3))
print("10,2,3:    ",min_time_for_copies(10, 2, 3))

n - кол-во копий 
x - время на первом ксероксе 
y - время на втором ксероксе

n,x,y:      min t
-----------------
0,3,5:      0
1,3,5:      3
1,5,3:      3
2,4,4:      8
3,4,4:      8
5,2,2:      6
4,1,100:    4
3,2,3:      5
10,2,3:     14


### Пояснение

Сначала мы тратим $min(x, y)$ минут на создание первой копии самым быстрым ксероксом, после чего используем бинарный поиск для поиска минимального времени $T$ на изготовление оставшихся n−1 копий.

В процессе поиска для каждого значения mid мы проверяем условие достаточности: суммарное количество копий, которое оба ксерокса могут напечатать за это время (mid // x + mid // y), должно быть не меньше $n-1$.

Алгоритм работает за $O(\log(\min(x,y) \times N))$.

## 3. Накормить животных

В зоопарке есть животные, каждое из которых требует определённое количество еды (целое число).
Каждая привезённая порция еды может накормить только одно животное и не делится.
Необходимо определить, сколько животных можно накормить имеющимся набором порций.

Что нужно сделать:
Реализуйте жадный алгоритм распределения еды (например, сортировка животных и порций). Объясните, почему выбранная стратегия даёт максимальное количество накормленных животных.

In [5]:
def feed_animals(animals: list[int], portions: list[int]) -> tuple[int, list[tuple]]:
    animals_sorted  = sorted(enumerate(animals),  key=lambda x: x[1])
    portions_sorted = sorted(enumerate(portions), key=lambda x: x[1])

    fed_count = 0
    fed_pairs = []          # (индекс животного, индекс порции)

    portion_ptr = 0         # указатель на текущую порцию

    for animal_idx, animal_need in animals_sorted:
        # Ищем наименьшую порцию, которой хватит этому животному
        while portion_ptr < len(portions_sorted):
            p_idx, p_size = portions_sorted[portion_ptr]
            portion_ptr += 1
            if p_size >= animal_need:          # порция подходит
                fed_count += 1
                fed_pairs.append((animal_idx, p_idx))
                break
            # порция слишком мала — пропускаем её (она никому не поможет)

    return fed_count, fed_pairs

In [6]:
animals  = [4, 2, 7, 1, 5]
portions = [3, 6, 2, 8, 1]

count, pairs = feed_animals(animals, portions)

print("Накормлено:", count, "из", len(animals))

for a_i, p_i in pairs:
    print("Животное", a_i+1, "съело порцию", p_i+1)

Накормлено: 4 из 5
Животное 4 съело порцию 5
Животное 2 съело порцию 3
Животное 1 съело порцию 2
Животное 5 съело порцию 4


### Пояснение

Сортируем животных и порции по возрастанию, затем каждому животному даём наименьшую подходящую порцию.
Это оптимально, потому что если кормить сначала самых "дешёвых" животных наименьшими подходящими порциями — крупные порции остаются для требовательных животных. Тратить большую порцию на маленькое животное расточительно — она могла бы накормить кого-то ещё.

Алгоритм работает за $O(n log_n + m log_m)$

## 4. Найти разницу между двумя строками

Даны строки a и b.
Строка b получена из строки a путём перемешивания символов и добавления одной новой буквы.
Необходимо вернуть эту добавленную букву.

Что нужно сделать:
Реализуйте решение без сортировки строк (желательно за O(n)). Объясните, как используется подсчёт символов или XOR для нахождения лишнего элемента.

In [7]:
from collections import Counter

def find_added_letter(a: str, b: str) -> str:
    return list(Counter(b) - Counter(a))[0]

In [8]:
print("str1  ->  str2      letter")
print("---------------------------")
print("abc   ->  cbad     ",find_added_letter("abc",   "cbad")  )
print("aab   ->  aabb     ",find_added_letter("aab",   "aabb")  )
print("xyz   ->  zyxx     ",find_added_letter("xyz",   "zyxx")  )
print("      ->  a        ",find_added_letter("",      "a")     )
print("hello ->  helloz   ",find_added_letter("hello", "helloz"))

str1  ->  str2      letter
---------------------------
abc   ->  cbad      d
aab   ->  aabb      b
xyz   ->  zyxx      x
      ->  a         a
hello ->  helloz    z


### Пояснение

Создаёт счётчики букв для обеих строк, вычитает один из другого, оставляя только лишнюю букву в b, и возвращает её за $O(2n) = O(n)$.

## 5. Сумма двух элементов массива

Дан неотсортированный массив целых чисел и число target.
Необходимо найти два элемента массива, сумма которых равна target.
Каждый элемент можно использовать только один раз.
Если подходящей пары нет — вернуть пустой массив.

Что нужно сделать:
Реализуйте эффективное решение за O(n), используя хеш-таблицу (словарь). Поясните, как хранение уже просмотренных элементов позволяет избежать двойного цикла.

In [9]:
def two_sum(nums: list[int], target: int) -> list[int]:
    seen = {}

    for i, x in enumerate(nums):
        complement = target - x
        if complement in seen:
            return [seen[complement], i]
        seen[x] = i

    return []

In [10]:

print("nums             target    result")
print("----------------------------------")
print("[2, 7, 11, 15]     22     ", two_sum([2, 7, 11, 15],22))
print("[3, 2, 4]           7     ", two_sum([3, 2, 4],      7))
print("[1, 5, 3, 7]        8     ", two_sum([1, 5, 3, 7],   8))
print("[1, 2, 3, 4]       10     ", two_sum([1, 2, 3, 4],  10))

nums             target    result
----------------------------------
[2, 7, 11, 15]     22      [1, 3]
[3, 2, 4]           7      [0, 2]
[1, 5, 3, 7]        8      [1, 2]
[1, 2, 3, 4]       10      []


### Пояснение
Со словарём вместо вложенного цикла получаем поиск за O(1) (спасибо хеш-таблицам)